# Gold performance: liquid clustering and OPTIMIZE

Run this after the pipeline has loaded the Gold layer (`run_pipeline`).

The fact and the aggregate tables are created with `CLUSTER BY` (liquid clustering). The keys
come from how the dashboard reads the data: almost every visual is filtered on one indicator,
and then on a year range, a group or a set of countries.

| Table | Clustering keys | Typical filter |
|---|---|---|
| `gold_fact_indicator_values` | `indicator_sk`, `year_key` | one indicator, a range of years |
| `gold_agg_indicator_trends` | `indicator_sk`, `group_type` | one indicator, regions or income groups |
| `gold_agg_yoy_growth` | `indicator_sk`, `country_sk` | one indicator, selected countries |
| `gold_agg_latest_values` | `indicator_sk` | one indicator, all countries (map / ranking) |

The dimensions have at most a few hundred rows, so they are not clustered: Spark broadcasts
them to every task in a join. The pipeline runs `OPTIMIZE` at the end of every Gold refresh
that changed data (logged as `OPTIMIZE gold tables`), which clusters the newly written files.

In [0]:
%run ./pipeline_utils

In [0]:
import time
from functools import reduce

tables = get_tables("", DEFAULT_SCHEMA)
LARGE = ["fact", "agg_trends", "agg_yoy", "agg_latest"]


def table_details(keys):
    return reduce(lambda a, b: a.unionByName(b), [
        spark.sql(f"DESCRIBE DETAIL {tables[k]}").select("name", "clusteringColumns", "numFiles", "sizeInBytes")
        for k in keys
    ])


display(table_details(GOLD_KEYS))

### 1. OPTIMIZE
`OPTIMIZE` on a liquid clustered table only rewrites files that are not clustered yet, so running
it again right after the pipeline usually changes nothing. The output shows the files removed
and added.

In [0]:
before = {r["name"]: r["numFiles"] for r in table_details(LARGE).collect()}
for key in LARGE:
    display(spark.sql(f"OPTIMIZE {tables[key]}"))

for r in table_details(LARGE).collect():
    print(f"{r['name']}: {before[r['name']]} files -> {r['numFiles']} files")

### 2. Table history
Each incremental run is one `MERGE` commit on the fact, followed by the `OPTIMIZE` commit of the
pipeline. `operationMetrics` shows the rows and files of every commit.

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {tables['fact']}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.col("version").desc())
)

### 3. Data skipping
Delta keeps min / max statistics per file for the clustering keys. A query filtered on one
indicator only reads the files whose `indicator_sk` range contains it, the others are skipped.
The cell below counts in how many files the rows of one indicator are stored.

With about 1.2 million rows the fact is still a small table, so after `OPTIMIZE` it can be only a few
files and the effect is small. The benefit grows with the number of files.

In [0]:
indicator_sk = (spark.table(tables["dim_indicator"])
                .filter("indicator_code = 'EG.ELC.ACCS.ZS'").first()["indicator_sk"])
fact = spark.table(tables["fact"])
total_files = fact.select("_metadata.file_path").distinct().count()
indicator_files = fact.filter(F.col("indicator_sk") == indicator_sk).select("_metadata.file_path").distinct().count()
print(f"Rows of EG.ELC.ACCS.ZS are stored in {indicator_files} of {total_files} files")

### 4. Star schema query, as the dashboard runs it
Average access to electricity per region and year. The filter on the indicator and the years
uses the clustering keys of the fact. Open the query profile of this cell to see the files read
and pruned by the scan of `gold_fact_indicator_values`.

In [0]:
query = f"""
    SELECT c.region, f.year_key, ROUND(AVG(f.value), 2) AS avg_access_pct, COUNT(f.value) AS countries
    FROM {tables['fact']} f
    JOIN {tables['dim_country']} c ON c.country_sk = f.country_sk
    JOIN {tables['dim_indicator']} i ON i.indicator_sk = f.indicator_sk
    WHERE i.indicator_code = 'EG.ELC.ACCS.ZS'
      AND f.year_key BETWEEN 2000 AND 2024
      AND c.entity_type = 'Country'
    GROUP BY c.region, f.year_key
    ORDER BY c.region, f.year_key
"""
start = time.time()
rows = spark.sql(query).collect()
print(f"{len(rows)} rows in {time.time() - start:.2f} s")
display(spark.sql(query))

### 5. The same question from the aggregate table
`gold_agg_indicator_trends` already holds the result per region and year, so the dashboard reads
a few hundred pre-computed rows instead of joining and grouping the fact.

In [0]:
query = f"""
    SELECT t.group_name AS region, t.year_key, ROUND(t.avg_value, 2) AS avg_access_pct,
           t.reporting_countries AS countries
    FROM {tables['agg_trends']} t
    JOIN {tables['dim_indicator']} i ON i.indicator_sk = t.indicator_sk
    WHERE i.indicator_code = 'EG.ELC.ACCS.ZS'
      AND t.group_type = 'Region'
      AND t.year_key BETWEEN 2000 AND 2024
    ORDER BY region, t.year_key
"""
start = time.time()
rows = spark.sql(query).collect()
print(f"{len(rows)} rows in {time.time() - start:.2f} s")
display(spark.sql(query))